# 03.01 · Datos del IDO para el insumo de Evidencia + Recomendaciones

Primer notebook del paso 3. Reúne, para los IDOs **PPI** del paso 2, todo lo que necesitan los
notebooks siguientes, sin llamar a Azure. El insumo solo se hace para operaciones PPI: un IDO
nuevo que pase por el paso 2 entra aquí automáticamente si es PPI.

- **Datos del IDO**, tomados de la tabla final del paso 2: ficha, montos, secciones 2.1 a 2.3,
  3, 4.1 y 5.1, intervención redactada y concepto de priorización.
- **Evidencia del repositorio sin resumir**: el texto completo de las columnas R (contenido de
  la evidencia), S (recomendaciones) y T (referencias) de las filas que eligió el `02.03`.
  En el Input CCI esa evidencia quedó sintetizada; el insumo parte de las celdas originales.

Las filas elegidas se pueden cambiar a mano para un IDO. Se generan dos tablas en
`data/interim/3. Insumo Evid+Recom/03.01_datos_ido/`:

| Archivo | Contenido |
|---|---|
| `idos_insumo.csv` | Una fila por IDO, con todas las columnas del paso 2 más las filas del repositorio que usará el insumo |
| `filas_repositorio.csv` | Una fila por IDO y fila del repositorio, con el texto completo de R, S y T |

## 1. LOAD LIBRARIES

Se usan pandas para las tablas y openpyxl para leer el repositorio. Las rutas se toman de
`caf_input_cci_preparatoria/config.py`.

In [ ]:
from pathlib import Path
import sys

from IPython.display import display
from openpyxl import load_workbook
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))  # para importar el paquete caf_input_cci_preparatoria

from caf_input_cci_preparatoria import config

## 2. LOAD DATA

### Rutas

In [2]:
archivo_idos = (
    config.INTERIM / "2. IDOs - Input CCI" / "02.04_concepto_priorizacion" / "idos_concepto.csv"
)
archivo_repositorio = config.RAW / "1. Protocolo de evidencia.xlsx"
carpeta_salida = config.INTERIM / "3. Insumo Evid+Recom" / "03.01_datos_ido"

### Tabla final del paso 2

`idos_concepto.csv` es la salida del `02.04`: una fila por IDO con todo lo extraído y redactado
en el paso 2. Se lee como texto para no alterar ningún valor. El paso 3 no ejecuta nada del paso 2:
si un IDO nuevo no aparece aquí, primero hay que correr el `02.00`.

In [3]:
idos = pd.read_csv(archivo_idos, encoding="utf-8-sig", dtype=str, keep_default_na=False)
print(f"{len(idos)} IDOs en el paso 2")
display(idos[["archivo", "producto_agrupado", "gran_sector", "filas_elegidas"]])

15 IDOs en el paso 2


,archivo,producto_agrupado,gran_sector,filas_elegidas
0,10 09 2026 - IDO PAED Barbados Programa Seguri...,PBL,SEGURIDAD,15
1,10 14 2026 - Borrador1 IDO ICE FR 228.docx,Línea de crédito,ELECTRIFICACIÓN,"38, 49, 59"
2,10 19 2026 - FR-228 - ANDE 8.docx,PPI,ELECTRIFICACIÓN,38
3,10 21 2026 - IDO Proyecto San Jacinto.docx,PPI,AGRICULTURA,"48, 61"
4,1errefref - CCI IDO 1806 PB GM FO l.docx,PPI,VIVIENDA,"13, 16"
5,BORRADOR CCI IDO_Programa OSE4-0.docx,PPI,AGUA Y SANEAMIENTO,24
6,FR 228 El Salvador Vuela II 21 9 2026 comparti...,PPI,TRANSPORTE,50
7,FR-228 IDO Bergantin_RD.docx,Otro,DESARROLLO PRODUCTIVO,
8,FR-228 guanay actualizado.docx,PPI,VIALIDAD,25
9,IDO Cierre de Brechas Chancay V18Ago2026 copy....,PPI,AGUA Y SANEAMIENTO,28


### Repositorio de evidencia

Se lee la hoja `(2) Repositorio evidencia` igual que en el `02.03`: encabezados en la fila 6 y
datos desde la fila 7; el número de fila del Excel es el índice. Además de las columnas que usa
el `02.03`, se toman:

| Columna | Campo | Para qué |
|---|---|---|
| Q | `insumo_previo` | Fecha y nombre de un insumo ya hecho con esa fila, como referencia |
| U | `confianza_detalle` | Puntaje y explicación del nivel de confianza |

In [4]:
columnas_repositorio = {
    "sector": 7,  # G
    "subsector": 8,  # H
    "nivel_confianza": 16,  # P
    "insumo_previo": 17,  # Q
    "contenido_evidencia": 18,  # R
    "recomendaciones": 19,  # S
    "referencias": 20,  # T
    "confianza_detalle": 21,  # U
}

libro = load_workbook(archivo_repositorio, read_only=True, data_only=True)
hoja = libro["(2) Repositorio evidencia"]
registros = []
for fila_excel, valores in enumerate(
    hoja.iter_rows(min_row=7, max_col=21, values_only=True), start=7
):
    registro = {
        campo: str(valores[columna - 1]).strip() if valores[columna - 1] is not None else ""
        for campo, columna in columnas_repositorio.items()
    }
    if any(registro.values()):
        registros.append({"fila": fila_excel, **registro})
libro.close()

repositorio = pd.DataFrame(registros).set_index("fila")
repositorio["tipo_intervencion"] = repositorio["sector"] + repositorio["subsector"].map(
    lambda subsector: f" - {subsector}" if subsector else ""
)
print(f"Filas del repositorio cargadas: {len(repositorio)}")

Filas del repositorio cargadas: 61


/Users/k/Library/CloudStorage/GoogleDrive-lbuitrago41@gmail.com/My Drive/General/6. Empleos/1. CAF/Asignaciones/2026/caf_input_cci_preparatoria/.venv/lib/python3.14/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


## 3. PROCESS DATA

### IDOs a los que se hace el insumo

Regla: el insumo se hace **solo para los IDOs PPI**, es decir, con `producto_agrupado` igual a
`PPI` en el paso 2 (el `02.04` agrupa ahí las distintas formas en que el IDO escribe la
modalidad). Los demás productos se listan como excluidos.

**Prueba con pocos IDOs:** si `idos_prueba` lleva nombres de archivo, solo esos IDOs pasan al
resto del paso 3 (`03.02` a `03.04`). Sirve para probar un cambio de reglas con uno o dos IDOs
antes de correr todos. Con la lista vacía (`[]`) pasan todos los PPI.
`idos_excluidos` hace lo contrario: deja por fuera los IDOs que se nombren (por ejemplo, para
procesarlos después como nuevos). Mientras el filtro está
activo, los CSV del `03.02` y del `03.03` guardan solo esos IDOs.

In [ ]:
es_ppi = idos["producto_agrupado"].eq("PPI")
seleccion = idos[es_ppi].copy()
idos_prueba = []  # [] = todos los IDOs PPI
idos_excluidos = [  # [] = ninguno
    # "FR-228 guanay actualizado.docx",
    # "CCI-IDO-Consolidación Plan INADEH v2.docx",
]
for lista in [idos_prueba, idos_excluidos]:
    assert set(lista) <= set(seleccion["archivo"]), f"No son PPI o no existen: {lista}"
if idos_prueba:
    seleccion = seleccion[seleccion["archivo"].isin(idos_prueba)]
    print(f"PRUEBA: solo {len(seleccion)} IDO(s) pasan al resto del paso 3")
if idos_excluidos:
    seleccion = seleccion[~seleccion["archivo"].isin(idos_excluidos)]
    print(f"Excluidos a mano: {'; '.join(idos_excluidos)}")
seleccion.index = pd.RangeIndex(1, len(seleccion) + 1)  # numeración desde 1
print(f"{len(seleccion)} IDOs PPI seleccionados")
print("Excluidos por no ser PPI:")
display(idos.loc[~es_ppi, ["archivo", "producto_agrupado"]])

12 IDOs PPI seleccionados
Excluidos por no ser PPI:


,archivo,producto_agrupado
0,10 09 2026 - IDO PAED Barbados Programa Seguri...,PBL
1,10 14 2026 - Borrador1 IDO ICE FR 228.docx,Línea de crédito
7,FR-228 IDO Bergantin_RD.docx,Otro


### Concepto de priorización sin aprobar

La sección 3 del insumo usa la decisión y el concepto finales que se aprueban o editan en el
`02.00` (`estado_concepto` Aprobado, Editado o Previo). Si un IDO PPI quedó **Pendiente**, el
Word mostrará el borrador de Azure marcado como tal; para evitarlo, se corre de nuevo el
`02.00` y se aprueba su concepto antes de este paso.

In [ ]:
pendientes = seleccion[seleccion["estado_concepto"].eq("Pendiente")]
print(f"{len(pendientes)} IDOs PPI con el concepto sin aprobar")
if len(pendientes):
    display(pendientes[["archivo", "estado_concepto"]])

### Filas del repositorio de cada IDO

Por defecto se usan las `filas_elegidas` del `02.03`. Para cambiarlas en un IDO, se agrega su
nombre en `ajustes_filas` con la lista completa de filas del Excel que debe usar (reemplaza a
las elegidas). La columna `origen_filas` indica de dónde salió cada lista.

In [6]:
ajustes_filas = {
    # "FR-228 guanay actualizado.docx": [25, 41],
}

In [7]:
def a_lista(texto):
    return [int(numero) for numero in texto.split(",") if numero.strip()]


ajenos = sorted(set(ajustes_filas) - set(seleccion["archivo"]))
if ajenos:
    raise ValueError(f"Ajustes para IDOs no seleccionados: {ajenos}")
inexistentes = sorted(
    {fila for filas in ajustes_filas.values() for fila in filas} - set(repositorio.index)
)
if inexistentes:
    raise ValueError(f"Filas que no existen en el repositorio: {inexistentes}")

filas_insumo = [
    ajustes_filas.get(archivo, a_lista(elegidas))
    for archivo, elegidas in zip(seleccion["archivo"], seleccion["filas_elegidas"])
]
seleccion["filas_insumo"] = [", ".join(map(str, filas)) for filas in filas_insumo]
seleccion["origen_filas"] = seleccion["archivo"].map(
    lambda archivo: "Ajuste manual" if archivo in ajustes_filas else "02.03"
)

### Evidencia completa de las filas

Se arma la tabla larga: una fila por cada IDO y fila del repositorio, con el texto de las
celdas tal cual.

In [8]:
campos_fila = ["tipo_intervencion", "nivel_confianza", "confianza_detalle", "insumo_previo"]
campos_texto = ["contenido_evidencia", "recomendaciones", "referencias"]

filas_repositorio = pd.DataFrame(
    [
        {"archivo": archivo, "fila": fila, **repositorio.loc[fila, campos_fila + campos_texto]}
        for archivo, filas in zip(seleccion["archivo"], filas_insumo)
        for fila in filas
    ],
    columns=["archivo", "fila", *campos_fila, *campos_texto],
)
print(f"{len(filas_repositorio)} filas del repositorio para {len(seleccion)} IDOs")

14 filas del repositorio para 12 IDOs


### Control de calidad

Por cada IDO se muestran sus filas, el nivel de confianza y el número de caracteres del texto
del repositorio y de las secciones del IDO que usarán los notebooks siguientes. Se avisa cuando:

- el IDO no tiene filas del repositorio;
- una fila no tiene contenido de evidencia (R);
- la sección 5.1 no tiene ninguna casilla marcada con ☒ (la plantilla quedó sin llenar).

In [9]:
secciones_ido = [
    "2.2_descripcion",
    "3_justificacion_beneficios",
    "4.1_riesgos",
    "5.1_riesgos_ambientales_sociales",
]
largos = filas_repositorio.groupby("archivo")[campos_texto].agg(
    lambda textos: textos.str.len().sum()
)
confianzas = filas_repositorio.groupby("archivo")["nivel_confianza"].agg(" | ".join)

vista = seleccion[["archivo", "producto_agrupado", "gran_sector", "filas_insumo"]].copy()
vista["confianza"] = vista["archivo"].map(confianzas).fillna("")
vista = vista.join(largos, on="archivo")
vista[secciones_ido] = seleccion[secciones_ido].map(len)
display(vista)

,archivo,producto_agrupado,gran_sector,filas_insumo,confianza,contenido_evidencia,recomendaciones,referencias,2.2_descripcion,3_justificacion_beneficios,4.1_riesgos,5.1_riesgos_ambientales_sociales
1,10 19 2026 - FR-228 - ANDE 8.docx,PPI,ELECTRIFICACIÓN,38,Alta,4837,2566,200,19944,8782,4121,14134
2,10 21 2026 - IDO Proyecto San Jacinto.docx,PPI,AGRICULTURA,"48, 61",Alta | Alta,15081,4973,691,4667,6785,1106,8536
3,1errefref - CCI IDO 1806 PB GM FO l.docx,PPI,VIVIENDA,"13, 16",Alta | Alta,11714,4815,3372,4696,7573,8554,1564
4,BORRADOR CCI IDO_Programa OSE4-0.docx,PPI,AGUA Y SANEAMIENTO,24,Media,0,5657,4139,7109,8540,9229,8714
5,FR 228 El Salvador Vuela II 21 9 2026 comparti...,PPI,TRANSPORTE,50,Media,3546,1673,1026,7602,7652,5689,11146
6,FR-228 guanay actualizado.docx,PPI,VIALIDAD,25,Alta,3410,3731,226,1885,6239,425,6918
7,IDO Cierre de Brechas Chancay V18Ago2026 copy....,PPI,AGUA Y SANEAMIENTO,28,Alta,6765,2919,216,10681,9230,4700,8331
8,IDO Cierre de Brechas Chancay V18Ago2026.docx,PPI,AGUA Y SANEAMIENTO,28,Alta,6765,2919,216,10681,9230,4700,8331
9,IDO Corredor Cusco - Choquequirao 22 10 2026.docx,PPI,VIALIDAD,25,Alta,3410,3731,226,12870,10595,1300,12026
10,IDO Dirección General de Aviación Civil (marzo...,PPI,TRANSPORTE,50,Media,3546,1673,1026,1380,8188,3478,10240


In [10]:
avisos = [
    f"{archivo}: sin filas del repositorio"
    for archivo, filas in zip(seleccion["archivo"], filas_insumo)
    if not filas
]
avisos += [
    f"{fila.archivo}: la fila {fila.fila} no tiene contenido de evidencia"
    for fila in filas_repositorio.itertuples()
    if not fila.contenido_evidencia
]
avisos += [
    f"{archivo}: sección 5.1 sin casillas marcadas"
    for archivo, texto in zip(seleccion["archivo"], seleccion["5.1_riesgos_ambientales_sociales"])
    if "☒" not in texto
]
print("\n".join(avisos) or "Sin avisos.")

BORRADOR CCI IDO_Programa OSE4-0.docx: la fila 24 no tiene contenido de evidencia
1errefref - CCI IDO 1806 PB GM FO l.docx: sección 5.1 sin casillas marcadas


### Revisar un IDO

Se cambia `ido_a_revisar` (número de la tabla anterior) para leer la justificación del `02.03`
y el texto completo del repositorio que recibirá el `03.02`.

In [11]:
ido_a_revisar = 1

ido = seleccion.loc[ido_a_revisar]
print(f"{ido['archivo']}\nFilas: {ido['filas_insumo']} ({ido['origen_filas']})\n")
print(f"Justificación del 02.03:\n{ido['justificacion_seleccion']}\n")
for fila in filas_repositorio[filas_repositorio["archivo"].eq(ido["archivo"])].itertuples():
    print(f"===== Fila {fila.fila}: {fila.tipo_intervencion} · confianza {fila.nivel_confianza}")
    print(f"Insumo previo: {fila.insumo_previo or 'ninguno'}")
    for campo in campos_texto:
        texto = getattr(fila, campo)
        print(f"\n--- {campo} ({len(texto)} caracteres)\n{texto}")

10 19 2026 - FR-228 - ANDE 8.docx
Filas: 38 (02.03)

Justificación del 02.03:
La fila 38 se selecciona por sus recomendaciones específicas sobre suministro de equipos, servidumbres e indemnizaciones, contratación de obras, supervisión y gestión social y ambiental, directamente pertinentes para la línea, las subestaciones y sus conexiones. También aborda reservas y seguimiento de la ejecución. Sus hallazgos, sin embargo, corresponden al acceso a electricidad y no evalúan ampliaciones de transmisión de 500 kV; por ello, se presentan como antecedentes, no como resultados atribuibles a esta operación. Ninguna candidata aporta contenido específico sobre auditoría externa, comisión de financiamiento, evaluación del préstamo o gestión de género. La fila 49 reitera recomendaciones pertinentes, pero no añade cobertura y sus hallazgos solares no corresponden a lo financiado. Se cita la síntesis de Impacto CAF con autor institucional; las recomendaciones de la fila 38 no tienen atribución individ

## 4. SAVE DATA

Se guardan las dos tablas en `data/interim/3. Insumo Evid+Recom/03.01_datos_ido/` (la carpeta
se crea si no existe). Se reescriben completas en cada ejecución.

| Archivo | Columnas |
|---|---|
| `idos_insumo.csv` | Todas las de `idos_concepto.csv` más `filas_insumo` y `origen_filas` |
| `filas_repositorio.csv` | `archivo`, `fila`, `tipo_intervencion`, `nivel_confianza`, `confianza_detalle`, `insumo_previo`, `contenido_evidencia`, `recomendaciones`, `referencias` |

El Excel del repositorio y la tabla del paso 2 no se modifican.

In [12]:
carpeta_salida.mkdir(parents=True, exist_ok=True)
seleccion.to_csv(carpeta_salida / "idos_insumo.csv", index=False, encoding="utf-8-sig")
filas_repositorio.to_csv(
    carpeta_salida / "filas_repositorio.csv", index=False, encoding="utf-8-sig"
)
print(
    f"{len(seleccion)} IDOs y {len(filas_repositorio)} filas guardados en "
    f"{carpeta_salida.relative_to(ROOT)}"
)

12 IDOs y 14 filas guardados en data/interim/3. Insumo Evid+Recom/03.01_datos_ido
